# Scaffolding project

_DSAIT4050: Information retrieval lecture, TU Delft_

Welcome to the **DSAIT4050: Information retrieval** lecture!

This project acts as a gentle introduction to information retrieval for you. You do not need any prior knowledge about IR for this task. Only some Python programming skills are required.

## Getting started

Under the hood, this notebook uses a library called **PyTerrier**. Please check out the first part of our _Introduction to PyTerrier_ series to learn how to install PyTerrier. However, you do not need to interact with PyTerrier directly for now; rather, we're providing you with simple utility functions you can use. Feel free to have a look how these are implemented, but it's not required.

**Task 1**: Install PyTerrier (see the `01-setup.ipynb` notebook).

Now you should be able to import the utility functions. A dataset will be downloaded and indexed automatically (this will take a minute).


In [1]:
pip install python-terrier==0.12.1

Note: you may need to restart the kernel to use updated packages.


In [3]:
from util import search, evaluate, evaluate_all, get_tf_idf

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
antique/test/non-offensive documents: 100%|██████████| 403666/403666 [00:00<00:00, 489715.39it/s]


Now that we have loaded the data, you can run search queries. For example:


In [3]:
search("what is the meaning of life")

00:01:05.963 [main] WARN org.terrier.querying.ApplyTermPipeline -- The index has no termpipelines configuration, and no control configuration is found. Defaulting to global termpipelines configuration of ''. Set a termpipelines control to remove this warning.


,qid,docid,docno,text,rank,score,query
0,1,284036,327334_2,If life did not suck sometime it would not be ...,0,25.241914,what is the meaning of life
1,1,24563,514843_5,"To live until we die,and have a meaningful. li...",1,24.428920,what is the meaning of life
2,1,183199,3286609_30,To make my mom and dad's life meaningful.,2,24.428920,what is the meaning of life
3,1,338875,1977360_8,"Change it to "" What makes life meaningful?""",3,24.428920,what is the meaning of life
4,1,146534,422602_7,to have a meaningful life or a life of meaning...,4,22.001176,what is the meaning of life
5,1,37272,3770850_1,"Oh... I get it.... ""thes""?. . Thes. is the abb...",5,21.882860,what is the meaning of life
6,1,142099,3352535_8,whats the purpose of life?,6,21.260102,what is the meaning of life
7,1,169083,2534143_13,whats life with no exitment!,7,21.260102,what is the meaning of life
8,1,338872,1977360_5,"Perhaps ""what can you do to make your life mor...",8,21.040540,what is the meaning of life
9,1,104996,4351526_3,the meaning of life is whatever you ascribe to...,9,19.934543,what is the meaning of life


What you get here is a list of ten documents from the corpus that are ordered by how relevant they are to our query (according to the search engine).

## Query rewriting

The goal of this task is to come up with a way of **rewriting queries** such that the search engine can "understand" them better.

In order to do this, let's first take a look at some example queries from our dataset. We represent these queries using a `pandas.DataFrame`, where the first column corresponds to the **query ID** and the second column corresponds to the **query**:


In [4]:
import pandas as pd

example_queries = pd.DataFrame(
    [
        [
            "443848",
            "does anybody know where i could get a free guide on how to train a siberian husky",
        ],
        [
            "1783010",
            "what is blaphsemy",
        ],
        [
            "2838988",
            "how can i get a cork out of not into a wine bottle without a corkscrew",
        ],
    ],
    columns=["qid", "query"],
)

Since these queries are taken from the dataset, we can **evaluate the performance** of our search engine on these queries. This means that we know which documents the system should retrieve for each query.

You can use the following evaluation function to do this. This function takes your queries and returns a score (mean average precision -- you will learn about this later). For now, all you need to know is that, the higher this score, the better the system works.

Let's evaluate the queries we have:


In [5]:
print("score:", evaluate(example_queries))

score: 0.07906002902973568


Now it's up to you to figure out if and how it's possible to make the search engine perform better on these queries. How would you query a search engine if you wanted to know about these topics? Experiment a bit.

**Task 2**: Try to manually come up with ways to rewrite or reformulate the queries so the performance improves.

**Important**: Make sure that the query IDs match! Otherwise, evaluation will not work.


In [6]:
example_queries_rewritten = pd.DataFrame(
    [
        # Write reflection about method used
        [
            "443848",
            # Original: does anybody know where i could get a free guide on how to train a siberian husky
            # Adding more general terms such as "Dog" helps find more relevant results
            "siberian husky dog train",
            #"siberian husky train dog",
        ],
        [
            "1783010",
            # Original: blaphsemy
            # Fix typos
            "what is Blasphemy",
            #"what Blasphemy is",
        ],
        [
            "2838988",
            # Original: how can i get a cork out of not into a wine bottle without a corkscrew
            "cork bottle without corkscrew",
            #"bottle corkscrew cork without",
        ],
    ],
    columns=["qid", "query"],
)
print("score after rewriting:", evaluate(example_queries_rewritten))

score after rewriting: 0.10694392789340007


TypeError: get_tf_idf() missing 1 required positional argument: 'doc_ids'

# An automatic approach

In this last part, we'll try to come up with an automatic approach to perform query re-writing. Use your findings from task 2 for this.

**Task 3**: Implement a function that automatically re-writes any input query.

You can use any approach or library you want for this task. However, keep in mind that simple ideas often work well!


In [32]:
from nltk import WordNetLemmatizer, word_tokenize
import re
from spellchecker import SpellChecker
import nltk
from nltk.corpus import stopwords, wordnet
from sentence_transformers import SentenceTransformer, util
import contractions

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('omw-1.4')
custom_stopwords = {'doe', 'th', 'w'}
stop_words = set(stopwords.words('english')).union(custom_stopwords)
spell = SpellChecker()
lemmatizer = WordNetLemmatizer()
bert_model = SentenceTransformer("all-MiniLM-L6-v2")

SIMILARITY_THRESHOLD = 0.9 #0.5


def remove_typos(query: str) -> str:
    """
    Remove typos using spellchecker.
    """
    # Tokenize the query
    words = word_tokenize(query)

    # # Lemmatize words
    words = [lemmatizer.lemmatize(word) for word in words]

    # Correct spelling
    corrected_words = [spell.correction(word) if spell.correction(word) else word for word in words]

    return " ".join(corrected_words)


def remove_stopwords(query: str) -> str:
    """
    Remove stop words from tokens.
    """
    tokens = word_tokenize(query)
    filtered = [token for token in tokens if not token in stop_words]
    filtered = filtered if len(filtered) > 0 else tokens
    return " ".join(filtered)


def normalize(query: str) -> str:
    tokens = word_tokenize(contractions.fix(query.replace("_", " ")))
    return " ".join(set(re.sub(r"[^a-zA-Z0-9']", '', token.replace('_', " ").lower()) for token in tokens))


def bert_similarity(word: str, query: str) -> float:
    """Compute cosine similarity using BERT sentence embeddings"""
    word_embedding = bert_model.encode(word, convert_to_tensor=True)
    query_embedding = bert_model.encode(query, convert_to_tensor=True)
    similarity_score = util.pytorch_cos_sim(word_embedding, query_embedding).item()
    return similarity_score


def add_hypernyms(query: str) -> str:
    query = normalize(query)
    tokens = word_tokenize(query)

    hypernyms = set()
    for token in tokens:
        synsets = wordnet.synsets(token)  # Get synsets for the word

        if not synsets:
            continue  # No synsets found

        hns = synsets[0].hypernyms()  # Get the first hypernym
        if not hns:
            continue
        for hn in hns:
            lemma = hn.lemmas()[0].name().replace('_', ' ')
            if bert_similarity(lemma, token) > SIMILARITY_THRESHOLD:
                hypernyms.add(lemma)

    tokens = list(hypernyms.union(tokens))
    return " ".join(tokens)


def rewrite_query(query: str) -> str:
    print("orig query:", query)
    operations = [normalize, remove_typos, remove_stopwords, add_hypernyms, normalize]
    for operation in operations:
        query = operation(query)

    print("rewriting query:", query)
    return query


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\maxde\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\maxde\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\maxde\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\maxde\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


This time, we'll evalute on _all_ queries in the dataset. This will give us a more general result:


In [8]:
print("score:", evaluate_all())

score: 0.06179994498738492


Are you able to improve the overall performance using your rewriting approach?


In [ ]:
print("score after rewriting", evaluate_all(rewrite_query))

orig query: how can we get concentration onsomething
rewriting query: something concentration get
orig query: why doesn t the water fall off earth if it s round
rewriting query: fall earth water round
orig query: how do i determine the charge of the iron ion in fecl3
rewriting query: charge iron determine feel ion
orig query: i have mice how do i get rid of them humanely
rewriting query: rid get mouse humanely
orig query: what does see leaflet mean on ept pregnancy test
rewriting query: mean test pregnancy eat leaflet see
orig query: what is innate immunity
rewriting query: innate immunity
orig query: how can i lose 30 pounds by june3
rewriting query: june 30 pound lose
orig query: what are the words to write the sound of raindrops moving train scribbling w pencil on paper figuratively
rewriting query: sound train write paper pencil figuratively word raindrop moving scribbling
orig query: why must i have an uncracked winshield in order to for my car to pass a safety inspeciton
rewritin